# PRDetect — GATv2 (Phan 4 Cai Tien) tren RAID Dataset

**Muc tieu:** Train va danh gia GATv2 (co che chu y dong) tren tap RAID.  
Notebook nay **chi chay GATv2** — khong so sanh GCN2.  

> **Chon GPU T4 x2 hoac P100** trong Kaggle Settings truoc khi chay!

## Buoc 1: Cai dat thu vien

In [ ]:
!pip install torch-geometric transformers datasets -q
!python -m spacy download en_core_web_sm -q
print('Cai dat hoan tat!')

In [ ]:
import torch
import torch_geometric
print(f'PyTorch : {torch.__version__}')
print(f'PyG     : {torch_geometric.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Buoc 2: Tai du lieu tu RAID (HuggingFace streaming)

In [ ]:
from datasets import load_dataset
import random
import time

# ------------------------------------------------------------------
# Tong so mau: 2000 Human + 2000 Machine = 4000 mau
# Tang len 3000/class neu muon ket qua tot hon (lau hon ~50%)
# ------------------------------------------------------------------
TARGET_PER_CLASS = 2000

print(f'Dang tai {TARGET_PER_CLASS*2} mau tu RAID (streaming)...')
t0 = time.time()

raid_stream = load_dataset('liamdugan/raid', split='train', streaming=True)

human_texts   = []
machine_texts = []
attack_stats  = {}
model_stats   = {}

for row in raid_stream:
    gen         = row.get('generation', '').strip()
    model_name  = row.get('model', 'unknown')
    attack_name = row.get('attack', 'none')

    # Loc: 15-300 tu, co noi dung thuc su
    words = gen.split()
    if not gen or len(words) < 15 or len(words) > 300:
        continue

    if model_name == 'human' and len(human_texts) < TARGET_PER_CLASS:
        human_texts.append(gen)
    elif model_name != 'human' and len(machine_texts) < TARGET_PER_CLASS:
        machine_texts.append(gen)
        attack_stats[attack_name] = attack_stats.get(attack_name, 0) + 1
        model_stats[model_name]   = model_stats.get(model_name, 0) + 1

    if len(human_texts) >= TARGET_PER_CLASS and len(machine_texts) >= TARGET_PER_CLASS:
        break

# Tron va chia train/val/test = 70/15/15
texts  = human_texts + machine_texts
labels = [1] * len(human_texts) + [0] * len(machine_texts)

combined = list(zip(texts, labels))
random.seed(2024)
random.shuffle(combined)
texts, labels = zip(*combined)
texts, labels = list(texts), list(labels)

n      = len(texts)
n_tr   = int(0.70 * n)
n_val  = int(0.15 * n)

train_texts,  train_labels  = texts[:n_tr],          labels[:n_tr]
val_texts,    val_labels    = texts[n_tr:n_tr+n_val], labels[n_tr:n_tr+n_val]
test_texts,   test_labels   = texts[n_tr+n_val:],    labels[n_tr+n_val:]

print(f'Xong! ({time.time()-t0:.0f}s)')
print(f'  Tong: {n} mau | Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')
print(f'  Cac kieu tan cong trong tap AI: {attack_stats}')
print(f'  Cac LLM: {list(model_stats.keys())}')

## Buoc 3: Xay dung Do thi Cu phap (SpaCy + RoBERTa)

In [ ]:
import spacy
import torch
from transformers import RobertaTokenizer, RobertaModel
from tqdm.notebook import tqdm

print('Nap SpaCy va RoBERTa...')
nlp    = spacy.load('en_core_web_sm')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

roberta_model     = RobertaModel.from_pretrained('roberta-base').to(device)
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base', do_lower_case=False)
roberta_model.eval()
print(f'Models loaded tren: {device}')

In [ ]:
def build_graph_data(texts, labels, desc='Building'):
    all_embeddings = []
    all_edge_index = []
    valid_labels   = []
    errors = 0

    for idx, text in enumerate(tqdm(texts, desc=desc)):
        try:
            doc    = nlp(text)
            tokens = [t.text for t in doc]
            if len(tokens) < 3:
                errors += 1; continue

            # RoBERTa embeddings (xu ly theo chunk 512)
            chunks = [tokens[i:i+512] for i in range(0, len(tokens), 512)]
            chunk_out = []
            for chunk in chunks:
                ids      = roberta_tokenizer.convert_tokens_to_ids(chunk)
                input_ids = torch.tensor(ids).unsqueeze(0).to(device)
                with torch.no_grad():
                    out = roberta_model(input_ids)
                chunk_out.append(out.last_hidden_state[0].cpu())
            embeddings = torch.cat(chunk_out, dim=0)

            # Dependency edges
            e0 = [w.i      for w in doc]
            e1 = [w.head.i for w in doc]
            edge_index = torch.tensor([e0, e1], dtype=torch.long)

            all_embeddings.append(embeddings)
            all_edge_index.append(edge_index)
            valid_labels.append(labels[idx])
        except Exception:
            errors += 1

    y = torch.tensor(valid_labels, dtype=torch.float32)
    print(f'  {desc}: OK={len(valid_labels)}, Errors={errors}')
    return {'all_token_embeddings': all_embeddings,
            'all_edge_index':       all_edge_index,
            'y':                    y}

t0 = time.time()
print('Bat dau build graph...')
train_g = build_graph_data(train_texts, train_labels, 'RAID Train')
val_g   = build_graph_data(val_texts,   val_labels,   'RAID Val  ')
test_g  = build_graph_data(test_texts,  test_labels,  'RAID Test ')
print(f'Hoan tat trong {(time.time()-t0)/60:.1f} phut!')

## Buoc 4: Dinh nghia GATv2 (Cai tien Phan 4)

4 diem khac biet so voi GCN2 goc:
- **[1] Trong so dong**: `alpha_ij = softmax(aT LeakyReLU(W[hi||hj]))` — khac voi GCN2 co dinh theo bac node
- **[2] Loc nhieu doi khang**: node bi tan cong (Homoglyph, Whitespace) bi gan `alpha~0` tu dong
- **[3] Multi-head**: 4 heads hoc 4 kieu tuong quan cu phap khac nhau
- **[4] ELU + tach dropout**: `attn_dropout=0.1` (attention) tach biet voi `dropout=0.5` (feature) + `LayerNorm`

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data

class GATv2Model(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads=4, dropout=0.5, attn_dropout=0.1):
        super(GATv2Model, self).__init__()

        # [1][2][3]: Dynamic attention, multi-head, adversarial filtering
        # attn_dropout (nho): regularize attention coefficient
        # dropout    (lon) : regularize node feature (tranh double-dropout)
        self.conv1 = GATv2Conv(
            in_channels=input_dim,
            out_channels=hidden_dim,
            heads=heads,
            dropout=attn_dropout,
            concat=True,
            add_self_loops=True,
            share_weights=False         # moi head hoc W rieng -> da goc nhin
        )
        self.conv2 = GATv2Conv(
            in_channels=hidden_dim * heads,
            out_channels=output_dim,
            heads=1,
            dropout=attn_dropout,
            concat=False,
            add_self_loops=True,
            share_weights=False
        )
        self.fc      = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout)

        # [4]: LayerNorm - on dinh gradient khi so node moi do thi khac nhau
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.norm2 = nn.LayerNorm(output_dim)

    def forward(self, data, return_attention=False):
        x, edge_index = data.x, data.edge_index

        # Layer 1: alpha_ij DONG theo (hi, hj) -> node nhieu bi alpha~0
        if return_attention:
            x, (_, alpha1) = self.conv1(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.elu(x)          # [4]: ELU thay ReLU
        x = self.dropout(x)   # [4]: feature dropout tach biet

        # Layer 2: van dynamic, khong ELU o layer cuoi
        if return_attention:
            x, (_, alpha2) = self.conv2(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = self.dropout(x)

        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)

        if return_attention:
            return torch.sigmoid(x), (alpha1, alpha2)
        return torch.sigmoid(x)

# Kiem tra so params
tmp = GATv2Model(768, 64, 64, heads=4)
print(f'GATv2Model tong params: {sum(p.numel() for p in tmp.parameters()):,}')
del tmp
print('Dinh nghia GATv2 hoan tat!')

## Buoc 5: Huan luyen GATv2 tren RAID

In [ ]:
import torch.optim as optim
import numpy as np
import os

# ---- Config ----
SEED       = 2024
EPOCHS     = 40
LR         = 0.0005
PATIENCE   = 7
HIDDEN_DIM = 64
OUTPUT_DIM = 64
HEADS      = 4
DROPOUT    = 0.5
ATTN_DROP  = 0.1
SAVE_PATH  = '/kaggle/working/gatv2_raid_best.pth'

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GATv2Model(
    input_dim    = 768,
    hidden_dim   = HIDDEN_DIM,
    output_dim   = OUTPUT_DIM,
    heads        = HEADS,
    dropout      = DROPOUT,
    attn_dropout = ATTN_DROP
).to(dev)

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
criterion = nn.BCELoss()

print(f'Config: hidden={HIDDEN_DIM}, heads={HEADS}, output={OUTPUT_DIM}')
print(f'        lr={LR}, dropout={DROPOUT}, attn_dropout={ATTN_DROP}')
print(f'        epochs={EPOCHS}, patience={PATIENCE}')
print(f'Device: {dev}')

In [ ]:
train_len = len(train_g['y'])
val_len   = len(val_g['y'])

best_val_acc  = -1
patience_cnt  = 0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

t_start = time.time()

for epoch in range(EPOCHS):

    # ===== Training =====
    model.train()
    e_loss, correct = 0.0, 0

    for i in tqdm(range(train_len), desc=f'Ep{epoch+1}/{EPOCHS} [Train]', leave=False):
        data = Data(
            x          = train_g['all_token_embeddings'][i],
            edge_index = train_g['all_edge_index'][i],
            y          = train_g['y'][i]
        ).to(dev)
        optimizer.zero_grad()
        out  = model(data)
        loss = criterion(out, data.y.float().view(-1, 1))
        loss.backward()
        optimizer.step()
        e_loss  += loss.item()
        correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()

    tr_loss = e_loss / train_len
    tr_acc  = correct / train_len

    # ===== Validation =====
    model.eval()
    e_loss, correct = 0.0, 0

    with torch.no_grad():
        for i in range(val_len):
            data = Data(
                x          = val_g['all_token_embeddings'][i],
                edge_index = val_g['all_edge_index'][i],
                y          = val_g['y'][i]
            ).to(dev)
            out  = model(data)
            loss = criterion(out, data.y.float().view(-1, 1))
            e_loss  += loss.item()
            correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()

    v_loss = e_loss / val_len
    v_acc  = correct / val_len

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(v_acc)

    tag = ''
    if v_acc >= best_val_acc:
        best_val_acc = v_acc
        patience_cnt = 0
        torch.save(model.state_dict(), SAVE_PATH)
        tag = ' <- BEST SAVED'
    else:
        patience_cnt += 1
        tag = f' ({patience_cnt}/{PATIENCE})'

    print(f'Ep{epoch+1:2d}: tr_acc={tr_acc:.4f} v_acc={v_acc:.4f}'
          f' | tr_loss={tr_loss:.4f} v_loss={v_loss:.4f}{tag}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping tai epoch {epoch+1}')
        break

elapsed = time.time() - t_start
print(f'\nHoan tat! Thoi gian: {elapsed/60:.1f} phut')
print(f'Best val_acc: {best_val_acc:.4f}')
print(f'Model luu tai: {SAVE_PATH}')

## Buoc 6: Danh gia tren RAID Test Set

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# Load best model
test_model = GATv2Model(
    input_dim=768, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM,
    heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROP
).to(dev)
test_model.load_state_dict(torch.load(SAVE_PATH, map_location=dev))
test_model.eval()

test_len   = len(test_g['y'])
preds_prob = []
correct    = 0

t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(test_len), desc='Testing'):
        data = Data(
            x          = test_g['all_token_embeddings'][i],
            edge_index = test_g['all_edge_index'][i],
            y          = test_g['y'][i]
        ).to(dev)
        out = test_model(data)
        preds_prob.append(out.item())
        correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()

y_pred = [1 if p >= 0.5 else 0 for p in preds_prob]
y_true = test_g['y'].numpy().tolist()

acc = correct / test_len
auc = roc_auc_score(y_true, preds_prob)
f1  = f1_score(y_true, y_pred)

print(f'\n===== KET QUA GATV2 TREN RAID TEST =====')
print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Test time : {time.time()-t0:.1f}s')
print('=======================================')
print()
print(classification_report(y_true, y_pred, target_names=['AI (0)', 'Human (1)']))

## Buoc 7: Bieu do Learning Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep = range(1, len(history['train_acc']) + 1)

axes[0].plot(ep, history['train_acc'], 'b--', alpha=0.6, label='Train Acc')
axes[0].plot(ep, history['val_acc'],   'b-',  lw=2,     label='Val Acc')
axes[0].axhline(y=acc, color='r', linestyle=':', lw=1.5, label=f'Test Acc={acc:.4f}')
axes[0].set_title('GATv2 — Accuracy tren RAID', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history['train_loss'], 'r--', alpha=0.6, label='Train Loss')
axes[1].plot(ep, history['val_loss'],   'r-',  lw=2,     label='Val Loss')
axes[1].set_title('GATv2 — BCE Loss tren RAID', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCE Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'GATv2 (Phan 4 Cai Tien) — RAID Dataset | Test Acc={acc:.4f} AUC={auc:.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/gatv2_raid_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Bieu do luu tai: /kaggle/working/gatv2_raid_curves.png')

## Buoc 8: (Tuy chon) Phan tich Attention — Phat hien Node Nhieu

Dung `return_attention=True` de xem GATv2 dang "nghi ngo" nhung node nao.

In [ ]:
# Lay 3 mau test bat ky de phan tich attention
test_model.eval()

for sample_idx in range(3):
    sample_data = Data(
        x          = test_g['all_token_embeddings'][sample_idx],
        edge_index = test_g['all_edge_index'][sample_idx],
        y          = test_g['y'][sample_idx]
    ).to(dev)

    with torch.no_grad():
        prob, (alpha1, alpha2) = test_model(sample_data, return_attention=True)

    # Tinh attention trung binh moi node nhan duoc
    mean_alpha = alpha1.mean(dim=-1)   # [E]
    edge_idx   = sample_data.edge_index
    N = sample_data.x.size(0)
    node_attn  = torch.zeros(N, device=dev)
    node_attn.scatter_add_(0, edge_idx[1], mean_alpha)
    deg        = torch.bincount(edge_idx[1], minlength=N).float().clamp(min=1)
    node_attn  = node_attn / deg

    suspicious = (node_attn < 0.05).sum().item()
    true_label = int(sample_data.y.item())
    pred_label = 1 if prob.item() >= 0.5 else 0

    print(f'Mau #{sample_idx}: Nhan that={"Human" if true_label==1 else "AI"} | '
          f'Du doan={"Human" if pred_label==1 else "AI"} (p={prob.item():.3f}) | '
          f'Node nhieu (attn<0.05): {suspicious}/{N}')

## Buoc 9: Luu Ket qua

In [ ]:
import pandas as pd
from datetime import datetime

result = {
    'model'   : 'GATv2_Phan4',
    'dataset' : 'RAID',
    'acc'     : round(acc, 4),
    'auc'     : round(auc, 4),
    'f1'      : round(f1, 4),
    'heads'   : HEADS,
    'hidden'  : HIDDEN_DIM,
    'dropout' : DROPOUT,
    'attn_dr' : ATTN_DROP,
    'lr'      : LR,
    'best_val': round(best_val_acc, 4),
    'time'    : datetime.now().strftime('%Y-%m-%d %H:%M')
}

df = pd.DataFrame([result])
df.to_csv('/kaggle/working/gatv2_raid_result.csv', index=False)

print('===== OUTPUT FILES =====')
print('  /kaggle/working/gatv2_raid_best.pth     <- model weights')
print('  /kaggle/working/gatv2_raid_curves.png   <- learning curves')
print('  /kaggle/working/gatv2_raid_result.csv   <- ket qua so')
print()
print(df.to_string(index=False))